In [1100]:
import sys
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append
from utils import *

In [1101]:
general_df = pd.read_csv('data/clean_data/cleaned_general_info.csv')
print(general_df.head())

  facility_id                    facility_name   city_town state  zip_code  \
0       10001  SOUTHEAST HEALTH MEDICAL CENTER      DOTHAN    AL     36301   
1       10005         MARSHALL MEDICAL CENTERS        BOAZ    AL     35957   
2       10006     NORTH ALABAMA MEDICAL CENTER    FLORENCE    AL     35630   
3       10007         MIZELL MEMORIAL HOSPITAL         OPP    AL     36467   
4       10011               ST. VINCENT'S EAST  BIRMINGHAM    AL     35235   

          hospital_type                           hospital_ownership  \
0  Acute Care Hospitals  Government - Hospital District or Authority   
1  Acute Care Hospitals  Government - Hospital District or Authority   
2  Acute Care Hospitals                                  Proprietary   
3  Acute Care Hospitals               Voluntary non-profit - Private   
4  Acute Care Hospitals               Voluntary non-profit - Private   

   hospital_overall_rating  
0                        3  
1                        2  
2          

In [1102]:
import sqlite3
import pandas as pd
import numpy as np
import tabulate

In [1103]:
#Create sample data for hospital info
hospitals_data = {
    'facility_id': [1, 2, 3],
    'facility_name': ['Hospital A', 'Hospital B', 'Hospital C'],
    'city_town': ['City X', 'City Y', 'City Z'],
    'state': ['State 1', 'State 2', 'State 3'],
    'zip_code': ['12345', '23456', '34567'],
    'hospital_type': ['Type 1', 'Type 2', 'Type 3'],
    'hospital_ownership': ['Ownership 1', 'Ownership 2', 'Ownership 3'],
    'hospital_overall_rating': [4, 3, 5]
}

#Create sample data for CEHRT
cehrt_data = {
    'facility_id': [1, 2, 3],
    'cehrt_id': ['CEHRT001', 'CEHRT002', 'CEHRT003'],
    'developer_name': ['Dev A', 'Dev B', 'Dev C'],
    'product_name': ['Product A', 'Product B', 'Product C'],
}

# Create DataFrames
hospital_df = pd.DataFrame(hospital_data)
cehrt_df = pd.DataFrame(cehrt_data)

In [1104]:
# Create SQLite database connection and tables
conn = sqlite3.connect("ehr_interoperability.db")
cursor = conn.cursor()

# Create tables hospital info
cursor.execute('''CREATE TABLE IF NOT EXISTS hospitals (
    facility_id INTEGER PRIMARY KEY,
    facility_name TEXT,
    city_town TEXT,
    state TEXT,
    zip_code TEXT,
    hospital_type TEXT,
    hospital_ownership TEXT,
    hospital_overall_rating INTEGER
);
''')

# Create tables for cehrt
cursor.execute('''CREATE TABLE IF NOT EXISTS cehrt (
    facility_id INTEGER,
    cehrt_id TEXT PRIMARY KEY,
    developer_name TEXT,
    product_name TEXT,
    FOREIGN KEY (facility_id) REFERENCES hospital_info (facility_id)
);
''')

#insert data into tables
hospital_df.to_sql("hospitals", conn, if_exists="replace", index=False)
cehrt_df.to_sql("cehrt", conn, if_exists="replace", index=False)

# Verify data insertion
print("Hospitals Table:\n")
print(pd.read_sql_query("SELECT * FROM hospital_info", conn))

print("\nCEHRT Table:\n")
print(pd.read_sql_query("SELECT * FROM cehrt", conn))

Hospitals Table:

   facility_id facility_name city_town    state zip_code hospital_type  \
0            1    Hospital A    City X  State 1    12345        Type 1   
1            2    Hospital B    City Y  State 2    23456        Type 2   
2            3    Hospital C    City Z  State 3    34567        Type 3   

  hospital_ownership  hospital_overall_rating  
0        Ownership 1                        4  
1        Ownership 2                        3  
2        Ownership 3                        5  

CEHRT Table:

   facility_id  cehrt_id developer_name product_name
0            1  CEHRT001          Dev A    Product A
1            2  CEHRT002          Dev B    Product B
2            3  CEHRT003          Dev C    Product C


In [1105]:
# hospital_df.to_markdown()
# cehrt_df.to_markdown()

In [1106]:
# Setting up the query function
def query(query: str):
    return pd.read_sql(query, conn)

In [1107]:
# Selecting all from table hospital_info
all = """SELECT * FROM hospitals"""
query(all)

,facility_id,facility_name,city_town,state,zip_code,hospital_type,hospital_ownership,hospital_overall_rating
0,1,Hospital A,City X,State 1,12345,Type 1,Ownership 1,4
1,2,Hospital B,City Y,State 2,23456,Type 2,Ownership 2,3
2,3,Hospital C,City Z,State 3,34567,Type 3,Ownership 3,5


In [1108]:
cehrt_df
# Selecting all from CEHRT table

,facility_id,cehrt_id,developer_name,product_name
0,1,CEHRT001,Dev A,Product A
1,2,CEHRT002,Dev B,Product B
2,3,CEHRT003,Dev C,Product C


In [1109]:
# Joining the tables
join = """
SELECT * 
FROM cehrt
LEFT JOIN hospitals
on hospitals.facility_id = cehrt.facility_id 
WHERE hospital_overall_rating >= 0;
"""
query(join)

,facility_id,cehrt_id,developer_name,product_name,facility_id,facility_name,city_town,state,zip_code,hospital_type,hospital_ownership,hospital_overall_rating
0,1,CEHRT001,Dev A,Product A,1,Hospital A,City X,State 1,12345,Type 1,Ownership 1,4
1,2,CEHRT002,Dev B,Product B,2,Hospital B,City Y,State 2,23456,Type 2,Ownership 2,3
2,3,CEHRT003,Dev C,Product C,3,Hospital C,City Z,State 3,34567,Type 3,Ownership 3,5


In [1110]:
join_df = hospital_df.merge(cehrt_df, left_on="facility_id", right_on="facility_id", how="left")
join_df

,facility_id,facility_name,city_town,state,zip_code,hospital_type,hospital_ownership,hospital_overall_rating,cehrt_id,developer_name,product_name
0,1,Hospital A,City X,State 1,12345,Type 1,Ownership 1,4,CEHRT001,Dev A,Product A
1,2,Hospital B,City Y,State 2,23456,Type 2,Ownership 2,3,CEHRT002,Dev B,Product B
2,3,Hospital C,City Z,State 3,34567,Type 3,Ownership 3,5,CEHRT003,Dev C,Product C


In [1111]:
# Perform the LEFT JOIN
joined_df = hospital_df.merge(
    cehrt_df,
    how="left",
    on ="facility_id"
)

In [ ]:

def compare_ratings_sqlite(general_info_path, cehrt_path, db_path='hospital_ehr.db'):
    """
    Loads cleaned hospital and CEHRT data into SQLite, performs a LEFT JOIN,
    and compares hospital ratings across EHR developers.

    Parameters:
    - general_info_path (str): Path to cleaned hospital general info CSV.
    - cehrt_path (str): Path to cleaned CEHRT CSV.
    - db_path (str): SQLite database file name (default: 'hospital_ehr.db').

    Returns:
    - pd.DataFrame: Summary of average rating and hospital count per developer.
    """
    # Load CSVs
    general_df = pd.read_csv(general_info_path)
    cehrt_df = pd.read_csv(cehrt_path)

    # Connect to SQLite
    conn = sqlite3.connect(db_path)

    # Insert into SQLite
    general_df.to_sql('hospital_info', conn, if_exists='replace', index=False)
    cehrt_df.to_sql('cehrt_info', conn, if_exists='replace', index=False)

    # Perform LEFT JOIN and query
    query = """
    SELECT 
        h.facility_id,
        h.hospital_overall_rating,
        c.developer_name
    FROM hospital_info h
    LEFT JOIN cehrt_info c
    ON h.facility_id = c.facility_id
    WHERE h.hospital_overall_rating IS NOT NULL AND c.developer_name IS NOT NULL
    """

    joined_df = pd.read_sql_query(query, conn)

    # Convert ratings to numeric
    joined_df['hospital_overall_rating'] = pd.to_numeric(joined_df['hospital_overall_rating'], errors='coerce')

    # Group and summarize
    summary = (
        joined_df
        .groupby('developer_name')
        .agg(
            average_rating=('hospital_overall_rating', 'mean'),
            hospital_count=('hospital_overall_rating', 'count')
        )
        .reset_index()
        .sort_values(by='average_rating', ascending=False)
    )

    conn.close()
    return summary      